## Preamble

In [ ]:
# Install packages if needed
# install.packages("tidyverse")
# install.packages("WDI")
# install.packages("RSQLite")

# Load libraries
library("tidyverse")
library("WDI")
library("RSQLite")

options(repr.plot.width = 12, repr.plot.height = 8)

In [ ]:
# Directory
setwd('/Users/tasia/Desktop/Data Analysis/Assessment')

### Load and Set Up GDP and Unemployment Data

In [ ]:
# Load data from database
conn <- dbConnect(SQLite(), "wdi_cross_sectional_gdp_unemployment_2023.db")

df_gdp <- dbGetQuery(conn, "SELECT * FROM gdp")
df_unp <- dbGetQuery(conn, "SELECT * FROM unemployment")

dbDisconnect(conn)

# Eyeball
head(df_gdp)
head(df_unp)

In [ ]:
# Convert columns to numeric and replace invalid values with Na
df_gdp$`2023 [YR2023]` <- as.numeric(na_if(df_gdp$`2023 [YR2023]`, ".."))
df_unp$`2023 [YR2023]` <- as.numeric(na_if(df_unp$`2023 [YR2023]`, ".."))

# Eyeball
head(df_gdp)
head(df_unp)

In [ ]:
# Remove rows with missing country codes
df_gdp <- filter(df_gdp, !is.na(`Country Code`), `Country Code` != "")
df_unp <- filter(df_unp, !is.na(`Country Code`), `Country Code` != "")

# Merge dataframes (inner join!)
df <- inner_join(df_gdp,
                 df_unp,
                 by = "Country Code")

# Eyeball
head(df)

In [ ]:
# Rename columns
df <- rename(df,
             gdp = `2023 [YR2023].x`,
             unemployment = `2023 [YR2023].y`)

# Eyeball
head(df)

In [ ]:
# Load country information
countries <- WDI_data$country

# Remove World Bank aggregates
countries <- filter(countries, region != "Aggregates")
df <- filter(df, `Country Code` %in% countries$iso3c)

# Remove missing values
df <- drop_na(df, gdp, unemployment)

# Add region information
df <- left_join(df,
                select(countries, iso3c, region),
                by = c("Country Code" = "iso3c"))

# Eyeball
head(df)

## Running 4 Regression Models

### Model 1: Simple Linear Regression

In [ ]:
# Regress unemployment on GDP per capita
model1 <- lm(unemployment ~ gdp, data = df)
summary(model1)

### Interpretation and Discussing the Output

Based on the Simple Linear Regression results in Model 1, GDP per capita is statistically significant at the 5% level, with a p-value of 0.0197. The coefficient for GDP per capita is -0.00004699, which means that, on average, a 1,000 dollar increase in GDP per capita is predicted to decrease the unemployment rate by approximately 0.047 percentage points.

The intercept (7.363) represents the predicted unemployment rate when GDP per capita is zero. While this value is statistically significant at the 1% level, its practical interpretation is limited since it corresponds to an unrealistic economic scenario.

Overall, Model 1 suggests a negative relationship between GDP per capita and unemployment. However, the R-squared value of 0.03086 indicates that GDP per capita explains only approximately 3.09% of the variation in unemployment rates across countries.

### Model 2: Linear-Log Regression

In [ ]:
# Regress unemployment on logged GDP per capita
model2 <- lm(unemployment ~ log(gdp), data = df)
summary(model2)

### Interpretation and Discussing the Output

Based on the Linear-Log Regression results in Model 2, the coefficient for logged GDP per capita is -0.2831, indicating that, on average, a 1% increase in GDP per capita is predicted to decrease the unemployment rate by approximately 0.00283 percentage points. However, with a p-value of 0.3341, the coefficient is not statistically significant at conventional significance levels.

The intercept (9.117) represents the predicted unemployment rate when logged GDP per capita is zero, i.e. when GDP per capita equals one dollar.

Overall, the logarithmic transformation of GDP per capita does not improve the explanatory power of the regression model, with the R-squared value decreasing to 0.005363 relative to Model 1.

### Model 3: Quadratic Regression

In [ ]:
# Regress unemployment on GDP per capita and GDP per capita squared
model3 <- lm(unemployment ~ gdp + I(gdp^2), data = df)
summary(model3)

### Interpretation and Discussing the Output

Based on the Quadratic Regression results in Model 3 above, the coefficient for GDP per capita is negative, whilst the coefficient for GDP per capita squared is positive, suggesting a potential non-linear relationship between GDP per capita and unemployment. However, neither coefficient is statistically significant at conventional significance levels, with respective p-values of 0.300 and 0.845.

The intercept (7.426) represents the predicted unemployment rate when GDP per capita is zero.

Overall, despite accounting for a potential non-linear relationship between the two variables, the inclusion of the quadratic term does not substantially improve the explanatory power of the regression model, with the R-squared value increasing only marginally to 0.03107 relative to Model 1.

### Model 4: Multiple Linear Regression with Regional Controls

In [ ]:
# Regress unemployment on GDP per capita and region
model4 <- lm(unemployment ~ gdp + region, data = df)
summary(model4)

### Interpretation and Discussing the Output

Based on the Multiple Linear Regression results in Model 4, GDP per capita is statistically significant at the 10% level, with a coefficient of -0.00003851. This means that, on average, a 1,000 dollar increase in GDP per capita is predicted to decrease the unemployment rate by approximately 0.039 percentage points, ceteris paribus.

The coefficients for Europe and Central Asia, and Latin America and the Caribbean are statistically significant at the 5% level, whilst those for the Middle East, North Africa, Afghanistan and Pakistan, and Sub-Saharan Africa are statistically significant at the 1% level. These regions have higher predicted unemployment rates relative to the reference region (East Asia and Pacific), holding GDP per capita constant.

The intercept (4.036) represents the predicted unemployment rate within the reference region when GDP per capita is zero.

Overall, the inclusion of regional controls improves the explanatory power of the regression model, with an R-squared value of 0.1127, indicating that approximately 11.27% of the variation in unemployment rates is explained by Model 4.